# Day 163 — Topic Modeling (LDA)
**Month 9 · NLP + Deep Learning · Google Colab**

| Item | Detail |
|---|---|
| Dataset | ReviewPulse India — 600 rows, seed=155 |
| Model | Latent Dirichlet Allocation (LDA) |
| Tasks | T1–T5 + Bonus |
| Total | 80 pts + 10★ bonus |
| GitHub | Month9-NLP-DeepLearning-Portfolio |

**NRA rule:** Print the table → read the exact value → write the insight. Never estimate from visuals.


## Section 1 — Raw Data
⛔ **DO NOT MODIFY THIS CELL**

In [2]:
# ── RAW DATA — DO NOT MODIFY ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(155)
n = 600

platforms     = rng.choice(['Upwork','Fiverr','Toptal','Freelancer','PeoplePerHour'], n, p=[0.30,0.25,0.20,0.15,0.10])
project_types = rng.choice(['Data Analysis','Dashboard','ML Model','Data Cleaning','Visualization'], n, p=[0.28,0.22,0.20,0.18,0.12])
ratings       = np.clip(rng.normal(4.1, 0.7, n), 1, 5).round(1)
hourly_rate   = rng.uniform(8, 85, n).round(2)
project_hrs   = rng.integers(5, 121, n)
client_region = rng.choice(['South Asia','North America','Europe','Southeast Asia','Middle East'], n, p=[0.30,0.25,0.20,0.15,0.10])
revisions     = rng.integers(0, 6, n)
response_hrs  = rng.uniform(0.5, 48, n).round(1)

positive_phrases = [
    "excellent work delivered on time","outstanding data insights provided","highly skilled analyst recommend",
    "great communication throughout project","perfect dashboard exceeded expectations","very professional quality work",
    "superb analysis clear visualizations","top notch data cleaning fast","brilliant ml model accurate results",
    "exceptional freelancer hire again definitely"
]
negative_phrases = [
    "poor quality work many errors","missed deadline project incomplete","terrible communication slow response",
    "wrong analysis needed complete redo","very disappointing results avoided","bad experience not recommend",
    "sloppy data cleaning many mistakes","inaccurate model predictions useless","late delivery unprofessional attitude",
    "wasted money avoid this freelancer"
]
neutral_phrases = [
    "average work meets basic requirements","okay results some improvements needed","decent analysis nothing special",
    "acceptable quality on time delivery","moderate skills standard output","fair work within budget constraints",
    "satisfactory results minor issues found","mediocre dashboard needs more polish","standard ml model average accuracy",
    "competent analyst reasonable communication"
]

review_texts = []
for r in ratings:
    if r >= 4.2:
        phrase = positive_phrases[rng.integers(0, len(positive_phrases))]
    elif r <= 2.8:
        phrase = negative_phrases[rng.integers(0, len(negative_phrases))]
    else:
        phrase = neutral_phrases[rng.integers(0, len(neutral_phrases))]
    review_texts.append(phrase)

hired_again = ((ratings >= 4.0) & (revisions <= 2) & (response_hrs <= 24)).astype(int)

df = pd.DataFrame({
    'platform': platforms, 'project_type': project_types, 'rating': ratings,
    'hourly_rate': hourly_rate, 'project_hours': project_hrs, 'client_region': client_region,
    'revisions': revisions, 'response_hours': response_hrs,
    'review_text': review_texts, 'hired_again': hired_again
})
print(f"Dataset loaded: {df.shape}")
print(df[['review_text','rating','hired_again']].head(5))


Dataset loaded: (600, 10)
                               review_text  rating  hired_again
0      acceptable quality on time delivery     3.9            0
1  satisfactory results minor issues found     3.5            0
2       sloppy data cleaning many mistakes     2.8            0
3      fair work within budget constraints     3.2            0
4    average work meets basic requirements     3.6            0


## Section 2 — Concept Notes

### What is Topic Modeling?

Topic Modeling is **unsupervised NLP** — it discovers hidden thematic structure in a corpus without labelled categories.

**LDA (Latent Dirichlet Allocation)** assumes:
- Each **document** is a mixture of topics (e.g., 70% positive-quality, 30% communication)
- Each **topic** is a distribution over words (e.g., "excellent", "recommend", "skilled" have high probability in the positive topic)

### Why LDA matters in freelance work
| Use Case | Business Value |
|---|---|
| Client review mining | Discover what clients actually care about |
| Support ticket analysis | Auto-tag tickets without labelled data |
| Market research reports | Find themes across 1000s of survey responses |
| Content strategy | Identify topic clusters for SEO/blog planning |

### Key parameters
| Parameter | Effect |
|---|---|
| `n_components` | Number of topics to discover |
| `random_state` | Reproducibility (always set = seed) |
| `max_iter` | Training iterations (20 is sufficient for short texts) |
| `learning_method` | `'batch'` = full corpus each iteration (better for small datasets) |

### How to evaluate LDA
- **Perplexity**: Lower is better — measures how well the model predicts held-out data
- **Log-likelihood**: Higher (less negative) is better
- **Human coherence**: Do the top words per topic make sense together?

### LDA vs Word/Sentence Embeddings (Day 161/162 bridge)
| | LDA | Sentence Embeddings |
|---|---|---|
| Output | Topic probability distribution | Dense vector |
| Supervision | Unsupervised | Unsupervised (pretrained) |
| Strength | Interpretable topics | Semantic similarity |
| Today's use | Discover hidden review themes | Previously: semantic search |

### NRA discipline (mandatory)
Every insight cell must follow:  
`Number` → exact computed value (print first, then write)  
`Reason` → causal mechanism (why this number exists)  
`Action` → specific committed action (no "consider", no "might")


## Section 3 — Practice Tasks

### Task 1 — Text Preprocessing + Document-Term Matrix (15 pts)

Build the document-term matrix (DTM) from `review_text` using `CountVectorizer`.

**Steps:**
1. Preprocess: lowercase + remove non-alpha characters → store in `df['clean_text']`
2. Define `stop_words` list: `['the','a','an','and','or','but','in','on','at','to','for','of','is','was','are','were','be','been','have','has','had']`
3. Fit `CountVectorizer(max_features=200, stop_words=stop_words, min_df=2)` on `clean_text`
4. Print DTM shape, vocab size, total tokens, avg tokens per doc (4 decimal places)

**Fill in the blanks:**
```python
def preprocess(text):
    text = ___________          # lowercase
    text = re.sub(___, '', text)  # keep only letters and spaces
    return text

cv = CountVectorizer(max_features=___, stop_words=___, min_df=___)
dtm = cv.fit_transform(___)
vocab = cv.get_feature_names_out()
```


In [3]:
# Part: Task 1 - Text Preprocessing + DTM
# Goal: Clean text, build document-term matrix with CountVectorizer.
# Method: Lowercase, remove non‑alpha chars, stopword filtering, min_df=2, max_features=200.

from sklearn.feature_extraction.text import CountVectorizer
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)   # keep only letters and spaces
    return text

df['clean_text'] = df['review_text'].apply(preprocess)

stop_words = ['the','a','an','and','or','but','in','on','at','to','for','of',
              'is','was','are','were','be','been','have','has','had']

cv = CountVectorizer(max_features=200, stop_words=stop_words, min_df=2)
dtm = cv.fit_transform(df['clean_text'])
vocab = cv.get_feature_names_out()

print(f"DTM shape: {dtm.shape}")          # (600, 91)
print(f"Vocab size: {len(vocab)}")
print(f"Total tokens: {dtm.sum()}")
avg_tokens = dtm.sum() / dtm.shape[0]
print(f"Avg tokens per doc: {avg_tokens:.4f}")

DTM shape: (600, 91)
Vocab size: 91
Total tokens: 2659
Avg tokens per doc: 4.4317


In [4]:
# T1 — NRA Insight (5 pts)
# Number: avg tokens per doc = 4.4317 ; vocab size = 91
# Reason: Short reviews (one phrase each) plus stopword removal and min_df=2 reduce vocabulary.
# Action: Use these sparse features (91 words) for LDA; k=4 is appropriate for short texts.

### Task 2 — Train LDA Model (20 pts)

Train an LDA model with **4 topics** on the DTM from T1.

**Specs:**
- `n_components=4`, `random_state=155`, `max_iter=20`, `learning_method='batch'`
- Print: doc_topic shape, perplexity (4 dp), log-likelihood (4 dp)
- `doc_topic` = 2D array where `doc_topic[i, j]` = probability that doc `i` belongs to topic `j`
- Each row sums to 1.0 — verify this: `assert abs(doc_topic[0].sum() - 1.0) < 1e-6`

**Fill in the blanks:**
```python
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=___, random_state=___,
                                 max_iter=___, learning_method=___)
doc_topic = lda.fit_transform(___)

print(f"Shape: {doc_topic.shape}")
print(f"Perplexity: {lda.perplexity(___):_.4f}")
print(f"Log-likelihood: {lda.score(___):_.4f}")
assert abs(doc_topic[0].sum() - 1.0) < 1e-6, "Row does not sum to 1"
```


In [5]:
# Part: Task 2 - Train LDA Model
# Goal: Fit LDA with 4 topics, compute perplexity and log‑likelihood.
# Method: Use LatentDirichletAllocation with batch learning, 20 iterations, seed=155.

from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=4, random_state=155,
                                max_iter=20, learning_method='batch')
doc_topic = lda.fit_transform(dtm)

print(f"doc_topic shape: {doc_topic.shape}")          # (600, 4)
print(f"Perplexity: {lda.perplexity(dtm):.4f}")       # 44.6456
print(f"Log-likelihood: {lda.score(dtm):.4f}")        # -10100.8942
assert abs(doc_topic[0].sum() - 1.0) < 1e-6, "Row does not sum to 1"
print("Assertion passed: row sums to 1.0")

doc_topic shape: (600, 4)
Perplexity: 44.6456
Log-likelihood: -10100.8942
Assertion passed: row sums to 1.0


In [6]:
# T2 — NRA Insight (5 pts)
# Number: perplexity = 44.6456
# Reason: Perplexity is 44.6 because the corpus consists of 30 repeating synthetic phrases; the high word repetition across documents gives LDA very consistent co-occurrence signals, reducing uncertainty per token.
# Action: In the bonus, I will compare perplexity for k=3,4,5 and choose the k with lowest perplexity, but always validate interpretability.

### Task 3 — Topic-Word Extraction + Naming (20 pts)

Extract and display the top 10 words for each of the 4 topics.

**Steps:**
1. Loop over `lda.components_` (shape: n_topics × vocab_size)
2. For each topic, `.argsort()[-10:][::-1]` gives the top-10 word indices (descending)
3. Map indices → words via `vocab`
4. Print the words, then **name each topic** based on the words you see

**Fill in the blanks:**
```python
print("Top 10 words per topic:")
for i, topic_vec in enumerate(lda.components_):
    top_idx = topic_vec.argsort()___[::-1]   # top 10 descending
    top_words = [vocab[j] for j in ___]
    print(f"Topic {i}: {top_words}")
```

**Topic naming cell** — fill in after seeing the words:
```python
topic_names = {
    0: "___",   # based on top words
    1: "___",
    2: "___",
    3: "___"
}
```


In [7]:
# Part: Task 3 - Topic-Word Extraction
# Goal: Display top 10 words per topic and assign human‑readable names.
# Method: argsort on component vectors, pick highest probability words.

print("Top 10 words per topic:")
for i, topic_vec in enumerate(lda.components_):
    top_idx = topic_vec.argsort()[-10:][::-1]   # top 10 descending
    top_words = [vocab[j] for j in top_idx]
    print(f"Topic {i}: {top_words}")

# Topic naming (based on observed word clusters)
topic_names = {
    0: "ML / Model Quality",          # ml, model, accurate, brilliant, standard
    1: "Communication & Process",     # communication, throughout, great, competent, reasonable
    2: "Strong Endorsement / Rehire", # hire, again, exceptional, definitely, recommend
    3: "Dashboard / Mediocre"         # mediocre, polish, needs, nothing, decent
}
print("Topic names:", topic_names)

Top 10 words per topic:
Topic 0: ['ml', 'model', 'standard', 'delivery', 'quality', 'acceptable', 'time', 'results', 'brilliant', 'accurate']
Topic 1: ['communication', 'project', 'throughout', 'great', 'competent', 'reasonable', 'analyst', 'results', 'very', 'needed']
Topic 2: ['work', 'freelancer', 'hire', 'exceptional', 'definitely', 'again', 'analyst', 'skilled', 'highly', 'recommend']
Topic 3: ['dashboard', 'analysis', 'polish', 'needs', 'more', 'mediocre', 'nothing', 'decent', 'special', 'work']
Topic names: {0: 'ML / Model Quality', 1: 'Communication & Process', 2: 'Strong Endorsement / Rehire', 3: 'Dashboard / Mediocre'}


In [8]:
# T3 — NRA Insight (5 pts)
# Number: topic 1 has the most interpretable word cluster (top words = ['communication','project','throughout','great','competent','reasonable','analyst','results','very','needed'])
# Reason: These words clearly cluster around communication quality and process reliability, which are distinct from technical quality or rehire intent. The co‑occurrence is driven by neutral/positive reviews that emphasise soft skills.
# Action: Use these topic names in a client deliverable to label review segments; e.g., "Communication Quality" vs "Technical Ability" vs "Rehire Endorsement".

In [9]:
# T3 — NRA Insight (5 pts)
# Number: topic 2 has the most interpretable word cluster (top words = ['work','freelancer','hire','exceptional','definitely','again','analyst','skilled','highly','recommend'])
# Reason: These words clearly signal a positive rehire sentiment; they naturally co‑occur in reviews that mention hiring the freelancer again.
# Action: Use this topic name ("Positive Rehire Potential") in a client deliverable to identify freelancers who are most likely to be rehired based on review language.

### Task 4 — Dominant Topic per Document (15 pts)

Assign each review to its most probable topic and analyse the distribution.

**Steps:**
1. `dominant_topic = doc_topic.argmax(axis=1)` → shape (600,)
2. Add to dataframe: `df['dominant_topic'] = dominant_topic`
3. Print topic value counts (how many reviews per topic)
4. Print avg max topic probability (how confidently is each review assigned?)
5. Show 2 sample reviews per topic

**Fill in the blanks:**
```python
dominant_topic = doc_topic.argmax(axis=___)
df['dominant_topic'] = ___

topic_counts = pd.Series(dominant_topic).value_counts().sort_index()
print("Docs per topic:
", ___)

avg_confidence = doc_topic.max(axis=1).___()
print(f"Avg max topic probability: {avg_confidence:.4f}")
```


In [10]:
# Part: Task 4 - Dominant Topic Assignment
# Goal: Assign each review to its highest‑probability topic and analyse confidence.
# Method: argmax over doc_topic probabilities; compute average maximum probability.

dominant_topic = doc_topic.argmax(axis=1)
df['dominant_topic'] = dominant_topic

topic_counts = pd.Series(dominant_topic).value_counts().sort_index()
print("Docs per topic:\n", topic_counts)   # 0:180, 1:159, 2:119, 3:142

avg_confidence = doc_topic.max(axis=1).mean()
print(f"Avg max topic probability: {avg_confidence:.4f}")   # 0.8570

# Sample reviews per topic
for i in range(4):
    print(f"\nTopic {i} samples:")
    samples = df[df['dominant_topic'] == i]['review_text'].values[:2]
    for s in samples:
        print("  -", s)

Docs per topic:
 0    180
1    159
2    119
3    142
Name: count, dtype: int64
Avg max topic probability: 0.8570

Topic 0 samples:
  - acceptable quality on time delivery
  - superb analysis clear visualizations

Topic 1 samples:
  - great communication throughout project
  - competent analyst reasonable communication

Topic 2 samples:
  - average work meets basic requirements
  - highly skilled analyst recommend

Topic 3 samples:
  - satisfactory results minor issues found
  - sloppy data cleaning many mistakes


In [11]:
# T4 — Sample reviews per topic (no blanks — write from scratch)
# Show 2 review_text examples for each topic 0-3
# Use: df[df['dominant_topic'] == i]['review_text'].values[:2]



In [12]:
# T4 — NRA Insight (5 pts)
# Number: avg max topic probability = 0.8570
# Reason: High confidence (0.857) indicates reviews are dominated by one clear theme, which is expected given the short, phrase‑based synthetic text.
# Action: In production, I would set a confidence threshold of 0.70; assignments below that would be flagged for manual review.

### Task 5 — Topic vs Hired Again Analysis (10 pts)

Cross-tabulate dominant topic with `hired_again` to identify which topics predict client return.

**Steps:**
1. Group by `dominant_topic`, aggregate `hired_again`: mean + count
2. Rename columns: `['hire_rate', 'count']`
3. Round to 4 decimal places, print the table
4. Identify best and worst topic by hire_rate
5. Write NRA insight

**Fill in the blanks:**
```python
topic_hire = df.groupby('dominant_topic')['hired_again'].agg([___, ___]).round(4)
topic_hire.columns = ['hire_rate', 'count']
print(topic_hire)

best_topic  = topic_hire['hire_rate'].idxmax()
worst_topic = topic_hire['hire_rate'].idxmin()
print(f"Best:  Topic {best_topic}  hire_rate={topic_hire.loc[best_topic,'hire_rate']:.4f}")
print(f"Worst: Topic {worst_topic} hire_rate={topic_hire.loc[worst_topic,'hire_rate']:.4f}")
```


In [13]:
# Part: Task 5 - Topic vs Hired Again
# Goal: Compute hire rate per dominant topic and identify best/worst.
# Method: Group by dominant_topic, aggregate mean and count.

topic_hire = df.groupby('dominant_topic')['hired_again'].agg(['mean', 'count']).round(4)
topic_hire.columns = ['hire_rate', 'count']
print("Hire rate by dominant topic:\n", topic_hire)

best_topic  = topic_hire['hire_rate'].idxmax()
worst_topic = topic_hire['hire_rate'].idxmin()
print(f"Best:  Topic {best_topic}  hire_rate={topic_hire.loc[best_topic,'hire_rate']:.4f}")
print(f"Worst: Topic {worst_topic} hire_rate={topic_hire.loc[worst_topic,'hire_rate']:.4f}")

Hire rate by dominant topic:
                 hire_rate  count
dominant_topic                  
0                  0.1056    180
1                  0.0943    159
2                  0.1849    119
3                  0.0493    142
Best:  Topic 2  hire_rate=0.1849
Worst: Topic 3 hire_rate=0.0493


In [14]:
# T5 — NRA Insight (5 pts)
# Number: Topic 2 hire_rate=0.1849 vs Topic 3 hire_rate=0.0493
# Reason: Topic 2 has the highest hire rate because its dominant words ("hire", "again", "exceptional", "definitely", "recommend") are only produced by high‑rated, low‑revision reviews — the same conditions that define hired_again=1. Topic 3's low rate reflects language associated with mediocrity/dissatisfaction.
# Action: A freelancer platform should recommend Topic 2 freelancers for repeat clients, and flag Topic 3 freelancers for coaching or further review.

### ★ Bonus — Perplexity Comparison: k = 3, 4, 5 (10★)

Train LDA with k=3, k=4, k=5 (same params, random_state=155) and compare perplexity + log-likelihood.

**Steps:**
1. Loop k in [3, 4, 5], train each model, collect perplexity and log-likelihood
2. Build a comparison DataFrame: columns = ['k', 'perplexity', 'log_likelihood']
3. Print the table
4. Write NRA: which k is best and why?

**Note:** Lower perplexity = better model fit. But always balance with human interpretability.


In [15]:
# Part: Bonus - Perplexity Comparison
# Goal: Train LDA with k=3,4,5 and compare perplexity and log‑likelihood.
# Method: Loop over k, fit LDA, collect scores, print table.

ks = [3, 4, 5]
results = []
for k in ks:
    lda_k = LatentDirichletAllocation(n_components=k, random_state=155,
                                      max_iter=20, learning_method='batch')
    lda_k.fit(dtm)
    perplexity = lda_k.perplexity(dtm)
    log_lik = lda_k.score(dtm)
    results.append({'k': k, 'perplexity': perplexity, 'log_likelihood': log_lik})

comp_df = pd.DataFrame(results)
print("Perplexity comparison:\n", comp_df.round(4))

Perplexity comparison:
    k  perplexity  log_likelihood
0  3     50.9726     -10453.2929
1  4     44.6456     -10100.8942
2  5     40.8799      -9866.5871


In [16]:
# Bonus — NRA Insight (5★ pts)
# Number: k=5 achieves lowest perplexity=40.8799
# Reason: More topics capture finer distinctions in the short reviews, reducing surprise (perplexity) from 50.97 (k=3) to 40.88 (k=5).
# Action: I would deploy k=4 for a client deliverable because it offers a good balance of interpretability (clear topic names) and low perplexity (44.65); k=5 may overfit and produce less stable topics.

## Section 4 — Scoring Rubric

| Task | Subtask | Pts |
|---|---|---|
| T1 | `preprocess()` correct (lowercase + regex) | 3 |
| T1 | CountVectorizer params correct (200, min_df=2, stop_words) | 2 |
| T1 | DTM shape (600, 91) | 3 |
| T1 | Total tokens=2659, avg=4.4317 | 2 |
| T1 | NRA — Number exact, Reason causal, Action committed | 5 |
| T2 | LDA params correct (4, seed=155, batch, 20 iter) | 3 |
| T2 | doc_topic shape (600,4), assert passes | 3 |
| T2 | Perplexity=44.6456, Log-likelihood=-10100.8942 | 4 |
| T2 | NRA — Number exact, Reason causal, Action committed | 5 |
| T2 | (missing assert pass) | -2 |
| T3 | Top words loop correct (argsort[-10:][::-1]) | 5 |
| T3 | Topic 0 & 1 words match key | 4 |
| T3 | Topic 2 & 3 words match key | 4 |
| T3 | Topic names reasonable (based on word profiles) | 2 |
| T3 | NRA — Number exact, Reason causal, Action committed | 5 |
| T4 | dominant_topic = argmax(axis=1) correct | 3 |
| T4 | Topic counts: 0→180, 1→159, 2→119, 3→142 | 4 |
| T4 | Avg max prob = 0.8570 | 3 |
| T4 | NRA — Number exact, Reason causal, Action committed | 5 |
| T5 | hire_rate table correct (all 4 values to 4dp) | 5 |
| T5 | best=Topic 2 (0.1849), worst=Topic 3 (0.0493) | 5 |
| T5 | NRA — Number exact, Reason causal, Action committed | 5 |
| **★ Bonus** | Loop k=3,4,5, perplexity values correct | 5★ |
| **★ Bonus** | NRA — k=5 lowest, k=4 practical, committed action | 5★ |
| **TOTAL** | | **80 + 10★** |

---
## Interview Framing
**Q: How does LDA differ from clustering (K-Means) for text analysis?**

*"Both are unsupervised, but K-Means gives each document exactly one cluster — hard assignment. LDA gives every document a probability distribution over topics — a review can be 60% about communication quality and 40% about technical delivery. That's much closer to how language actually works. In client work I'd use K-Means when I need clean segments for a dashboard, and LDA when I need nuanced theme extraction — like mining 10,000 support tickets to find what's really frustrating customers. The output of LDA also gives you human-readable topic labels from top words, which you can't get from centroid coordinates."*
